# 테스트 환경 설정

이 노트북을 **가장 먼저 실행**하세요.

각 Wave 노트북을 실행하기 전에 아래 셀들을 실행해야 합니다.

In [1]:
# 필요 패키지 설치
import subprocess
subprocess.run(['pip', 'install', 'ipytest', 'pytest', 'pytest-asyncio', 'openpyxl', '-q'], check=True)
print('패키지 설치 완료')

패키지 설치 완료


In [2]:
import sys
from pathlib import Path
from unittest.mock import MagicMock

# 프로젝트 루트 기준 경로 설정
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / 'backend'))
sys.path.insert(0, str(PROJECT_ROOT))

# 미설치 외부 패키지 mock 처리
_OPTIONAL_MODULES = [
    'langchain_openai',
    'langchain_anthropic',
    'docling',
    'docling.document_converter',
    'fitz',
]
for _mod in _OPTIONAL_MODULES:
    if _mod not in sys.modules:
        sys.modules[_mod] = MagicMock()

import ipytest
ipytest.autoconfig()

print('경로 설정 완료')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

경로 설정 완료
PROJECT_ROOT: C:\Users\User\Desktop\Project\Ontology-Driven-RAG-


In [4]:
# 공유 픽스처 정의
import json
import pytest
from langchain_core.documents import Document
from app.factories.config import Config, LLMConfig, EmbeddingConfig, VectorDBConfig, PromptConfig

SAMPLE_REGISTRY = {
    'tech_expert': {
        'persona': '당신은 기술 전문가입니다.',
        'guide': '정확하고 간결하게 답변하세요.',
    },
    'maintenance_expert': {
        'persona': '당신은 설비 유지보수 전문가입니다.',
        'guide': '안전 절차를 우선적으로 안내하세요.',
    },
}

@pytest.fixture
def ollama_config():
    return Config(
        id='TEST',
        llm=LLMConfig(provider='ollama', model_name='test-model', base_url='http://localhost:11434'),
        embedding=EmbeddingConfig(model='test-embed'),
        vector_db=VectorDBConfig(db_path='./test_data/chroma', retrieval_k=3, score_threshold=0.7),
        prompt=PromptConfig(system_prompt='당신은 테스트 전문가입니다.', prompt_id='tech_expert'),
    )

@pytest.fixture
def openai_config():
    return Config(
        id='TEST_OAI',
        llm=LLMConfig(provider='openai', model_name='gpt-4o'),
        embedding=EmbeddingConfig(model='test-embed', base_url='http://localhost:11434'),
        vector_db=VectorDBConfig(db_path='./test_data/chroma_oai'),
        prompt=PromptConfig(system_prompt='You are a test assistant.', prompt_id='tech_expert'),
    )

@pytest.fixture
def anthropic_config():
    return Config(
        id='TEST_ANT',
        llm=LLMConfig(provider='anthropic', model_name='claude-sonnet-4-6'),
        embedding=EmbeddingConfig(model='test-embed', base_url='http://localhost:11434'),
        vector_db=VectorDBConfig(db_path='./test_data/chroma_ant'),
        prompt=PromptConfig(system_prompt='You are a test assistant.', prompt_id='tech_expert'),
    )

@pytest.fixture
def registry_path(tmp_path):
    path = tmp_path / 'registry.json'
    path.write_text(json.dumps(SAMPLE_REGISTRY, ensure_ascii=False), encoding='utf-8')
    return str(path)

@pytest.fixture
def sample_docs():
    return [
        Document(page_content='섹션 A 내용입니다.', metadata={'source': 'test.pdf'}),
        Document(page_content='섹션 B 내용입니다.', metadata={'source': 'test.pdf'}),
    ]

@pytest.fixture
def upstage_api_response():
    return {
        'elements': [
            {'content': {'markdown': '## 제목\n내용 텍스트'}, 'type': 'heading', 'page': 1, 'confidence': 0.99},
            {'content': {'markdown': '단락 텍스트입니다.'}, 'type': 'paragraph', 'page': 1, 'confidence': 0.95},
            {'content': {'markdown': ''}, 'type': 'figure', 'page': 2, 'confidence': 0.8},
        ]
    }

print('공유 픽스처 정의 완료')

공유 픽스처 정의 완료
